In [3]:
from disk import make_disk
import k3d

plot = k3d.plot()
plot += make_disk([10.0, 10, 10], 1.0, color=0x4488ff, side="double")
plot.display()

Output()

In [4]:
from visualize import build_sphere
import k3d

plot = k3d.plot()
plot += build_sphere(radius=30.0, opacity=0.35, color=0x3366aa)
plot.display()

Output()

In [5]:
from visualize import build_sphere
import k3d

plot = k3d.plot()
plot += make_disk([10.0, 0, 0], 1.0, color=0x4488ff, side="double")
plot += build_sphere(radius=10.0, opacity=0.35, color=0x3366aa)
plot.display()

Output()

In [6]:
# В метрах
from detector import Detector
det = Detector.fibonacci_sphere(n_pmts=40, radius=2, pmt_radius=0.1, a_pmt=0.5)
det.draw(n_azi=192, n_polar=192).display()


Output()

In [7]:
print (f'{det.pmts[0]} * {len(det.pmts)}')

PMT(position=array([0.44385386, 1.94756097, 0.        ]), radius=0.1, direction=array([-0.22220486, -0.975     , -0.        ]), a_pmt=0.5) * 40


In [8]:
import numpy as np
import k3d
from detector import Detector

# Детектор: полая сфера R=2 м с 40 ФЭУ радиуса 0.1 м.
det = Detector.fibonacci_sphere(n_pmts=40, radius=2, pmt_radius=0.1, a_pmt=0.5)

# Направления к центрам ФЭУ (единичные) — из существующих pmt.
u = np.array([p.position / np.linalg.norm(p.position) for p in det.pmts])
ua, ub = u[0], u[5]                     # два «помеченных» ФЭУ: 0 и 5

# Точки на стенке сферы: направления из det.pmts, масштаб — радиус детектора.
on_sphere = lambda d: det.radius * d

# Угловой радиус зоны ФЭУ на сфере — берём из параметров детектора.
theta_max = np.arcsin(det.pmts[0].radius / det.radius)
axis = np.cross(ua, [0.0, 0.0, 1.0])
if np.linalg.norm(axis) == 0.0:
    axis = np.cross(ua, [1.0, 0.0, 0.0])
def rot(v, a, ang):                     # поворот вектора v вокруг оси a (Родригес)
    a = a / np.linalg.norm(a)
    return (np.cos(ang) * v + np.sin(ang) * np.cross(a, v)
            + (1 - np.cos(ang)) * np.dot(a, v) * a)

# Четыре точки на сфере: две в зоне pmt 0 (разные), одна в зоне pmt 5, антипод.
points = np.array([
    on_sphere(rot(ua, axis, 0.5 * theta_max)),   # в зоне pmt 0
    on_sphere(rot(ua, axis, -0.5 * theta_max)),  # в зоне pmt 0 (другая)
    on_sphere(ub),                               # в зоне pmt 5
    -on_sphere(ua),                              # антипод — вне всех зон
])

# Честный прогон определения.
res = det.find_pmt_batch(points)
for i, (pt, pid) in enumerate(zip(points, res)):
    print(f'точка {i} {pt} -> pmt {pid}')

# Отрисовка детектора с ФЭУ и точками (зелёные — в зоне, красная — вне).
plot = det.draw(n_azi=192, n_polar=192)
colors = [0x22dd22 if pid >= 0 else 0xdd2222 for pid in res]
plot += k3d.points(points, colors=colors, point_size=0.06, shader='3d')
plot.display()




точка 0 [0.44427073 1.94939015 0.05001564] -> pmt 0
точка 1 [ 0.44427073  1.94939015 -0.05001564] -> pmt 0
точка 2 [ 1.162271    1.45       -0.73934167] -> pmt 5
точка 3 [-0.44440972 -1.95       -0.        ] -> pmt -1


Output()

In [9]:
# Использование класса Source и отрисовка направлений фотонов в k3d.
import numpy as np
import k3d
from detector import Detector
from source import Source

# Детектор и источник фотонов.
det = Detector.fibonacci_sphere(n_pmts=40, radius=2, pmt_radius=0.1, a_pmt=0.5)
src = Source(Y=10000)                    # 10000 фотон/МэВ

# Разыгрываем направления фотонов для события энергией E (МэВ).
E = 15.0                                  # событие: 150000 фотонов
dirs = src.emit(E, seed=0)                # (n, 3) единичные направления из начала координат
print(f'E = {E} МэВ -> n = {len(dirs)} фотонов')

# Для наглядности покажем первые 2000 направлений: концы единичных векторов,
# спроецированные на стенку сферы детектора R.
n_show = min(2000, len(dirs))
pts = det.radius * dirs[:n_show]          # точки на стенке сферы (R = 2)

plot = det.draw(n_azi=192, n_polar=192)
plot += k3d.points(pts, color=0x22dd22, point_size=0.03, shader='3d')
plot.display()

E = 15.0 МэВ -> n = 150000 фотонов


Output()